In [36]:
# import 环境
# import gym
import gymnasium as gym
# import RL 算法
from stable_baselines3 import PPO
import numpy as np
from stable_baselines3.common.evaluation import evaluate_policy

# 可能需要 pip install shimmy

In [37]:
class MyWrapper(gym.Wrapper):
    def __init__(self):
        env = gym.make('CartPole-v1')
        super().__init__(env)
        self.env = env

    def reset(self, seed=None, options=None):
        state, _ = self.env.reset()
        return state, _

    def step(self, action):
        state, reward, done, _, info = self.env.step(action)
        return state, reward, done, _, info

env = MyWrapper()

env.reset()
env

<MyWrapper<TimeLimit<OrderEnforcing<PassiveEnvChecker<CartPoleEnv<CartPole-v1>>>>>>

In [38]:
model = PPO('MlpPolicy', env, verbose=0)
model

In [39]:
evaluate_policy(model, env, n_eval_episodes=20)

(np.float64(9.15), np.float64(0.6538348415311012))

In [40]:
# 训练 测试
model.learn(total_timesteps=2_0000, progress_bar=True)

Output()

In [41]:
evaluate_policy(model, env, n_eval_episodes=20)

D:\ProgramData\anaconda3\Lib\site-packages\stable_baselines3\common\evaluation.py:70: UserWarning: Evaluation environment is not wrapped with a ``Monitor`` wrapper. This may result in reporting modified episode lengths and rewards, if other wrappers happen to modify these. Consider wrapping environment first with ``Monitor`` wrapper.
  warnings.warn(


(np.float64(496.45), np.float64(15.47409124956939))

In [42]:
########## 训练&保存模型

In [43]:
model = PPO('MlpPolicy', env, verbose=0).learn(8000, progress_bar=True)

Output()

In [44]:
# 保存模型
model.save('models/save00')

D:\ProgramData\anaconda3\Lib\site-packages\stable_baselines3\common\save_util.py:284: UserWarning: Path 'models' does not exist. Will create it.
  warnings.warn(f"Path '{path.parent}' does not exist. Will create it.")


In [45]:
# 加载模型
model = PPO.load('models/save00')

In [46]:
model

In [47]:
### 3. 加载模型后继续训练

In [48]:
from stable_baselines3 import A2C

In [52]:
model = A2C('MlpPolicy', env, verbose=0, gamma=0.9, n_steps=20).learn(8000, progress_bar=True)

model.save('model/save')
model = A2C.load('model/save', verbose=0)

# 继续训练 ，需要重置 env, 因为 env 在保存模型时是不能保存下来的
model.set_env(env)
model.learn(8000, progress_bar= True)


Output()

Output()

In [53]:
# 4。 gym 环境的包装

In [55]:
class StepLimitWrapper(gym.Wrapper):
    def __init__(self, env):
        super().__init__(env)
        self.current_step = 0

    def reset(self, seed=None, options=None):
        self.current_step = 0
        state, _ = self.env.reset()
        return state, _

    def step(self, action):
        self.current_step += 1
        state, reward, done, _, info = self.env.step(action)

        if self.current_step >= 100:
            done = True
        return state, reward, done, _, info


In [ ]:
from stable_baselines3.common.env_util import make_vec_env

In [56]:
# 5. 游戏环境

In [ ]:
def test_env():
    env.observation_space # 状态空间
    env.action_space # 动作空间

    state = env.reset()
    action = env.action_space.sample()
    next_state, reward, done, _ = env.step(action)

    state
    action
    next_state
    reward
    done

In [57]:
pip install huggingface-sb3

Looking in indexes: https://pypi.mirrors.ustc.edu.cn/simple/Note: you may need to restart the kernel to use updated packages.

  Using cached https://mirrors.ustc.edu.cn/pypi/packages/b5/2c/e8c25087c6e43e5e68659f2a4651988831d4b3e70791c433537c90d147ce/huggingface_sb3-3.0-py3-none-any.whl (9.7 kB)
  Using cached https://mirrors.ustc.edu.cn/pypi/packages/cb/bd/1a875e0d592d447cbc02805fd3fe0f497714d6a2583f59d14fa9ebad96eb/huggingface_hub-0.36.0-py3-none-any.whl (566 kB)
  Using cached https://mirrors.ustc.edu.cn/pypi/packages/06/7c/34330a89da55610daa5f245ddce5aab81244321101614751e7537f125133/wasabi-1.1.3-py3-none-any.whl (27 kB)

   ------------- -------------------------- 1/3 [huggingface-hub]
   ------------- -------------------------- 1/3 [huggingface-hub]
   ------------- -------------------------- 1/3 [huggingface-hub]
   ---------------------------------------- 3/3 [huggingface-sb3]



In [58]:
from huggingface_sb3 import load_from_hub